# DATA LOADING PIPELINE (POLARS) - V4

Advanced pipeline with priority-based missing value handling for **LB TOP 5% GUARANTEED**

**Purpose:**
- Load datasets
- Apply priority-based missing value strategies
- cancelled - Create missing flags and filled columns
- Convert data types
- Save processed data for modeling

**Priority Strategy:**
1. **ULTRA Priority:** bz, aw, cc (Δw 10x + test 5%)
2. **HIGH Priority:** az, bl, l, m (Δy=18)
3. **TEST38% Priority:** w, x, y, z (test 38% missing)
4. **CORE Priority:** at, by, ay, cd, ce, cf, al (>5% train+test)
5. **LOW Priority:** Rest 30 features (<1%, safe ffill)

**Expected Result:** 144 new columns (48 flags + 48 filled) = 238 total columns

## 1.1 IMPORTS AND CONFIGURATION

In [1]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import polars as pl
from pathlib import Path
import time
import numpy as np

# Polars configuration
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

# Set up paths
base_dir = Path("..")
train_path = base_dir / "data" / "processed" / "train_processed_v1.parquet"  # Float32 without clipping
# Test data options - choose one:
# Option 1: Original test data (Float64)
#test_path = base_dir / "data" / "test.parquet"
# Option 2: Processed test data (Float32) - uncomment to use
test_path = base_dir / "data" / "processed" / "test_processed_v1.parquet"
processed_dir = base_dir / "data" / "processed"
results_dir = base_dir / "Results"
print("✅ Configuration complete")
print(f"📁 Base directory: {base_dir}")

✅ Configuration complete
📁 Base directory: ..


## 1.2 DATA LOADING

In [2]:
# ============================================
# LOAD DATASETS
# ============================================

print("="*60)
print("LOADING DATASETS")
print("="*60)

train_full = pl.read_parquet(train_path)
test_full = pl.read_parquet(test_path)

print(f"✅ Train loaded: {train_full.shape}")
print(f"✅ Test loaded: {test_full.shape}")

# szybki sanity check
print("\n📊 Dtypes (train):")
print(train_full.dtypes[:10])

LOADING DATASETS
✅ Train loaded: (5337414, 94)
✅ Test loaded: (1447107, 92)

📊 Dtypes (train):
[String, Categorical, Categorical, Categorical, Int16, Int16, Float32, Float32, Float32, Float32]


## 1.3 FEATURE GROUPS

In [3]:
# ============================================
# FEATURE GROUPS (PYTHONIC + CORRECT LOGIC)
# ============================================

FEATURE_GROUPS = {
    "ultra": ['feature_bz', 'feature_aw', 'feature_cc'],
    "high": ['feature_az', 'feature_bl', 'feature_l', 'feature_m'],
    "test38": ['feature_w', 'feature_x', 'feature_y', 'feature_z'],
    "core": ['feature_at', 'feature_by', 'feature_ay', 'feature_cd',
             'feature_ce', 'feature_cf', 'feature_al'],
}

# all feature columns
all_features = [c for c in train_full.columns if c.startswith("feature_")]

# detect missing features
missing_cols = [
    c for c in all_features
    if train_full[c].null_count() > 0
]

no_missing_cols = [
    c for c in all_features
    if train_full[c].null_count() == 0
]

# flatten defined groups
high_priority_features = [
    f for group in FEATURE_GROUPS.values() for f in group
]

# 🔥 FIX: LOW only from missing columns
priority_low = [
    f for f in missing_cols if f not in high_priority_features
]

# ============================================
# PRINT SUMMARY
# ============================================

print("🎯 Feature groups summary:")
for k, v in FEATURE_GROUPS.items():
    print(f"   {k.upper():<7} ({len(v)}): {v[:3]}{'...' if len(v)>3 else ''}")

print(f"\n📊 Total features: {len(all_features)}")
print(f"📉 With NaN: {len(missing_cols)}")
print(f"✅ Without NaN: {len(no_missing_cols)}")
print(f"🧊 LOW priority (only NaN features!): {len(priority_low)}")

🎯 Feature groups summary:
   ULTRA   (3): ['feature_bz', 'feature_aw', 'feature_cc']
   HIGH    (4): ['feature_az', 'feature_bl', 'feature_l']...
   TEST38  (4): ['feature_w', 'feature_x', 'feature_y']...
   CORE    (7): ['feature_at', 'feature_by', 'feature_ay']...

📊 Total features: 86
📉 With NaN: 48
✅ Without NaN: 38
🧊 LOW priority (only NaN features!): 30


## 2.1 MISSING FLAGS

## 2.2 FILL TEST38

In [4]:
# FILL TEST38 (NO LEAKAGE - GROUPED FORWARD FILL)
# ============================================

TEST38 = FEATURE_GROUPS["test38"]

# Grouped forward fill by categorical keys
train_full = train_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in TEST38
])

test_full = test_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in TEST38
    if c in test_full.columns
])

print(f"✅ TEST38 forward-filled (grouped by code/sub_code/sub_category): {len(TEST38)}")

✅ TEST38 forward-filled (grouped by code/sub_code/sub_category): 4


## 2.3 FILL CORE + LOW

In [5]:
# ============================================
# FILL CORE + LOW (NO LEAKAGE - GROUPED FORWARD FILL)
# ============================================

CORE = FEATURE_GROUPS["core"]
LOW = priority_low

fill_cols = CORE + LOW

# Grouped forward fill by categorical keys
train_full = train_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in fill_cols
])

test_full = test_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in fill_cols
    if c in test_full.columns
])

print(f"✅ CORE + LOW forward-filled (grouped by code/sub_code/sub_category): {len(fill_cols)}")

✅ CORE + LOW forward-filled (grouped by code/sub_code/sub_category): 37


## 2.5 ULTRA/HIGH FEATURES

**ULTRA Priority:** Δw 10x + test 5% missing - **Bayesian MCMC ready**

**HIGH Priority:** Δy=18 impact - **Bayesian MCMC ready**

In [9]:
# ============================================
# MCMC IMPUTATION WITH SAVED MODELS (SAMPLED)
# ============================================
print("="*60)
print("BAYESIAN IMPUTATION - ULTRA & HIGH (SAMPLED DATA)")
print("="*60)

from sklearn.linear_model import BayesianRidge
import numpy as np
import joblib
from pathlib import Path

ULTRA = ['feature_bz', 'feature_aw', 'feature_cc']
HIGH = ['feature_az', 'feature_bl', 'feature_l', 'feature_m']
MCMC_FEATURES = ULTRA + HIGH

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# ============================================
# STEP 1: TRAIN MODELS ON SAMPLED TRAIN DATA
# ============================================
print("\n📊 STEP 1: Training Bayesian models on sampled train data...")

models = {}

for feat in MCMC_FEATURES:
    print(f"  Training model for: {feat}")

    # Get rows where feature is not null
    train_clean = train_full.filter(pl.col(feat).is_not_null())

    if len(train_clean) < 100:
        print(f"    ⚠️  Not enough data → forward fill")
        models[feat] = None
        continue

    # 🔥 SAMPLE to reduce memory (use 500k rows max)
    sample_size = min(500_000, len(train_clean))
    train_sample = train_clean.sample(n=sample_size, seed=42)

    print(f"    Using {len(train_sample):,} rows (sampled from {len(train_clean):,})")

    # Select predictors: other features (without target, without weight)
    pred_cols = [c for c in train_full.columns
                 if c.startswith('feature_')
                 and c != feat
                 and c not in MCMC_FEATURES]

    # Add frequency encodings for categories
    for cat in ['code', 'sub_code', 'sub_category']:
        freq_col = f'{cat}_freq'
        freq_map = train_sample.group_by(cat).agg(pl.len().alias(freq_col)).to_dict(as_series=False)
        train_sample = train_sample.with_columns([
            pl.col(cat).replace_strict(freq_map[cat], freq_map[freq_col], default=0).alias(freq_col)
        ])
        pred_cols.append(freq_col)

    # Add temporal features
    train_sample = train_sample.with_columns([
        (pl.col('ts_index') / 3600).alias('ts_norm'),
        (pl.col('horizon') / 25).alias('horizon_norm')
    ])
    pred_cols.extend(['ts_norm', 'horizon_norm'])

    # Prepare data
    X_train = train_sample.select(pred_cols).to_numpy()
    y_train = train_sample.select(feat).to_numpy().ravel()

    # Handle potential NaNs
    if np.any(np.isnan(X_train)):
        X_train = np.nan_to_num(X_train, nan=0.0)

    # Train Bayesian Ridge
    if feat in HIGH:
        # Higher precision for HIGH (Δy=18)
        model = BayesianRidge(max_iter=300, tol=1e-5, alpha_1=1e-5, lambda_1=1e-5)
    else:
        # Standard for ULTRA
        model = BayesianRidge(max_iter=200, tol=1e-4, alpha_1=1e-6, lambda_1=1e-6)

    model.fit(X_train, y_train)

    # Save model and predictors
    models[feat] = {
        'model': model,
        'pred_cols': pred_cols,
        'is_high': feat in HIGH
    }

    # Save to disk
    joblib.dump(model, model_dir / f"{feat}_model.pkl")
    print(f"    ✅ Model saved")

# ============================================
# STEP 2: APPLY TO TEST
# ============================================
print("\n📊 STEP 2: Applying Bayesian imputation to test data...")

for feat in MCMC_FEATURES:
    print(f"  Imputing: {feat}")

    if models[feat] is None:
        test_full = test_full.with_columns([
            pl.col(feat).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(feat)
        ])
        continue

    null_mask = test_full[feat].is_null()

    if null_mask.sum() == 0:
        print(f"    ✅ No nulls in test")
        continue

    # Prepare predictors for test
    test_temp = test_full.clone()

    for cat in ['code', 'sub_code', 'sub_category']:
        freq_col = f'{cat}_freq'
        if freq_col not in test_temp.columns:
            train_freq = train_full.group_by(cat).agg(pl.len().alias(freq_col))
            test_temp = test_temp.join(train_freq, on=cat, how='left')
            test_temp = test_temp.with_columns(pl.col(freq_col).fill_null(0))

    test_temp = test_temp.with_columns([
        (pl.col('ts_index') / 3600).alias('ts_norm'),
        (pl.col('horizon') / 25).alias('horizon_norm')
    ])

    pred_cols = models[feat]['pred_cols']
    X_test = test_temp.filter(null_mask).select(pred_cols).to_numpy()

    if np.any(np.isnan(X_test)):
        X_test = np.nan_to_num(X_test, nan=0.0)

    y_pred = models[feat]['model'].predict(X_test)

    # Create DataFrame with predictions
    null_ids = test_temp.filter(null_mask).select('id').to_numpy().ravel()
    pred_df = pl.DataFrame({
        'id': null_ids,
        f'{feat}_pred': y_pred
    })

    # Join and fill nulls
    test_full = test_full.join(pred_df, on='id', how='left')
    test_full = test_full.with_columns([
        pl.col(feat).fill_null(pl.col(f'{feat}_pred')).alias(feat)
    ])
    test_full = test_full.drop(f'{feat}_pred')

    print(f"    ✅ Imputed {null_mask.sum():,} rows")

# ============================================
# STEP 3: CLEANUP
# ============================================
print("\n📊 STEP 3: Cleaning up temporary columns...")

temp_cols = [c for c in test_full.columns if c.endswith('_freq') or c in ['ts_norm', 'horizon_norm']]
test_full = test_full.drop(temp_cols)

print(f"✅ Bayesian imputation complete for {len(MCMC_FEATURES)} features")
print(f"   ULTRA (3): {ULTRA}")
print(f"   HIGH (4): {HIGH}")


📊 STEP 2: Applying Bayesian imputation to test data...
  Imputing: feature_bz
    ✅ No nulls in test
  Imputing: feature_aw
    ✅ No nulls in test
  Imputing: feature_cc
    ✅ No nulls in test
  Imputing: feature_az
    ✅ No nulls in test
  Imputing: feature_bl
    ✅ No nulls in test
  Imputing: feature_l
    ✅ No nulls in test
  Imputing: feature_m
    ✅ No nulls in test
✅ Bayesian imputation complete


## 2.6 SAVE PROCESSED DATA

In [13]:
# ============================================
# SAVE PROCESSED DATA - V4 (MCMC IMPUTATION)
# ============================================
print("="*60)
print("SAVING PROCESSED DATA - V4 MCMC PIPELINE")
print("="*60)

# Create directory
processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Define paths
train_out_path = processed_dir / "train_processed_v5.parquet"
test_out_path = processed_dir / "test_processed_v5.parquet"

print("💾 Saving enhanced datasets...")

# Save files
train_full.write_parquet(train_out_path)
test_full.write_parquet(test_out_path)

print(f"✅ Train saved: {train_out_path}")
print(f"✅ Test saved: {test_out_path}")

# Get file sizes
train_size = train_out_path.stat().st_size / 1024**2
test_size = test_out_path.stat().st_size / 1024**2

print(f"\n📁 Files created:")
print(f"   Train: {train_size:.1f} MB")
print(f"   Test: {test_size:.1f} MB")

# Summary for v4
feature_cols_count = len([c for c in train_full.columns if c.startswith('feature_')])
print(f"\n🎯 V4 MCMC Pipeline Complete!")
print(f"\n📈 Pipeline Summary:")
print(f"   Total columns: {train_full.shape[1]}")
print(f"   Feature columns: {feature_cols_count}")
print(f"   Imputation methods:")
print(f"      • Bayesian MCMC (ULTRA + HIGH): {len(ULTRA) + len(HIGH)} features")
print(f"      • Forward fill (TEST38, CORE, LOW): {len(TEST38) + len(CORE) + len(LOW)} features")
print(f"\n🚀 v4 ready for LightGBM training!")
print(f"💡 Expected: Better score than v3")

SAVING PROCESSED DATA - V4 MCMC PIPELINE
💾 Saving enhanced datasets...
✅ Train saved: ..\data\processed\train_processed_v4.parquet
✅ Test saved: ..\data\processed\test_processed_v4.parquet

📁 Files created:
   Train: 414.3 MB
   Test: 69.9 MB

🎯 V4 MCMC Pipeline Complete!

📈 Pipeline Summary:
   Total columns: 94
   Feature columns: 86
   Imputation methods:
      • Bayesian MCMC (ULTRA + HIGH): 7 features
      • Forward fill (TEST38, CORE, LOW): 41 features

🚀 v4 ready for LightGBM training!
💡 Expected: Better score than v3


## 4.1 DIMENSIONALITY REDUCTION (SVD)

**Purpose:** Reduce 134 features → 50 components for numerical stability

**Implementation:**
- Fit SVD on training data
- Transform both train and test
- Create state vectors: [y_target, svd_1, ..., svd_50]

In [ ]:
# ============================================
# STEP 5: SVD REDUCTION (134 → 50)
# ============================================
print("="*60)
print("STEP 5: SVD DIMENSIONALITY REDUCTION")
print("="*60)

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Get all feature columns
feature_cols = [c for c in train_full.columns if c.startswith('feature_')]
print(f"Original features: {len(feature_cols)}")

# Prepare data for SVD
X_train_full = train_full.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_full)

# PCA to 50 components (more stable than SVD directly)
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Reduced to: {X_pca.shape[1]} components")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.4f}")

# Create PCA features for train
pca_cols = [f'pca_{i}' for i in range(50)]
train_full = train_full.with_columns([
    pl.Series(name=pca_cols[i], values=X_pca[:, i]) for i in range(50)
])

# For test
X_test_full = test_full.select(feature_cols).to_numpy()
X_test_scaled = scaler.transform(X_test_full)
X_test_pca = pca.transform(X_test_scaled)

test_full = test_full.with_columns([
    pl.Series(name=pca_cols[i], values=X_test_pca[:, i]) for i in range(50)
])

print(f"✅ PCA features added to train and test")

## 5.1 ESTIMATE A-MATRIX PER GROUP

**Purpose:** Estimate transition matrix A for each hierarchical level

**Implementation:**
- For each combination of (code, sub_code, sub_category, horizon)
- Build X = state vector
- Estimate A from regression: ΔX = A * X
- Solution: X(t+h) = e^(A * h) * X(t)

In [ ]:
# ============================================
# STEP 6: ESTIMATE MATRIX A PER GROUP
# ============================================
print("="*60)
print("STEP 6: ESTIMATING STATE TRANSITION MATRIX A")
print("="*60)

from scipy.linalg import expm
from sklearn.linear_model import LinearRegression

# State vector: y_target + PCA components
state_cols = ['y_target'] + pca_cols
print(f"State vector size: {len(state_cols)}")

# Store models per group
A_matrices = {}

# Group by code, sub_code, sub_category, horizon
groups = train_full.group_by(['code', 'sub_code', 'sub_category', 'horizon'])

for (code, sub, cat, h), group in groups:
    group = group.sort('ts_index')
    if len(group) < 20:  # Not enough data
        continue

    # Build state matrix X (rows = time steps)
    X_state = group.select(state_cols).to_numpy()

    # Compute ΔX = X_{t+1} - X_t
    X_t = X_state[:-1]
    X_t1 = X_state[1:]
    dX = X_t1 - X_t

    # Estimate A: dX = A * X_t
    reg = LinearRegression(fit_intercept=False)
    reg.fit(X_t, dX)
    A = reg.coef_.T  # dX = A @ X_t

    # Store
    key = f"{code}_{sub}_{cat}_{h}"
    A_matrices[key] = {
        'A': A,
        'last_state': X_state[-1],
        'horizon': h
    }

    if len(A_matrices) % 100 == 0:
        print(f"  Processed {len(A_matrices)} groups")

print(f"✅ Estimated A matrices for {len(A_matrices)} groups")

## 7 HIERARCHICAL A-MATRIX

**Purpose:** Combine hierarchical A matrices with weighted averaging

**Implementation:**
- A_code – dynamics at code level
- A_sub – dynamics at sub_code level
- A_cat – dynamics at category level
- Weighted average: A_final = α*A_code + β*A_sub + γ*A_cat

In [ ]:
# ============================================
# STEP 7: HIERARCHICAL MATRIX A (CODE → SUB → CAT)
# ============================================
print("="*60)
print("STEP 7: BUILDING HIERARCHICAL STATE TRANSITION")
print("="*60)

# ============================================
# 7.1 Aggregate A matrices by hierarchy level
# ============================================

# Store A matrices per level
A_by_code = {}
A_by_sub = {}
A_by_cat = {}

for key, data in A_matrices.items():
    parts = key.split('_')
    code = parts[0]
    sub = parts[1]
    cat = parts[2]
    horizon = int(parts[3])

    A = data['A']

    # Aggregate by code
    if code not in A_by_code:
        A_by_code[code] = []
    A_by_code[code].append(A)

    # Aggregate by sub_code
    sub_key = f"{code}_{sub}"
    if sub_key not in A_by_sub:
        A_by_sub[sub_key] = []
    A_by_sub[sub_key].append(A)

    # Aggregate by category
    if cat not in A_by_cat:
        A_by_cat[cat] = []
    A_by_cat[cat].append(A)

# Compute mean A per level
print("Computing mean A per hierarchy level...")

A_code_mean = {}
for code, matrices in A_by_code.items():
    A_code_mean[code] = np.mean(matrices, axis=0)
    print(f"  Code {code}: {len(matrices)} matrices → mean A")

A_sub_mean = {}
for sub_key, matrices in A_by_sub.items():
    A_sub_mean[sub_key] = np.mean(matrices, axis=0)

A_cat_mean = {}
for cat, matrices in A_by_cat.items():
    A_cat_mean[cat] = np.mean(matrices, axis=0)
    print(f"  Category {cat}: {len(matrices)} matrices → mean A")

print(f"✅ Hierarchical A matrices computed:")
print(f"   Code level: {len(A_code_mean)} unique codes")
print(f"   Sub-code level: {len(A_sub_mean)} unique sub-codes")
print(f"   Category level: {len(A_cat_mean)} unique categories")

# ============================================
# 7.2 Weighted combination for each group
# ============================================

# Hyperparameters for hierarchical weights (tunable)
# Higher weight for more specific level
ALPHA_CODE = 0.3   # weight for code-level dynamics
ALPHA_SUB = 0.5    # weight for sub-code level (most specific)
ALPHA_CAT = 0.2    # weight for category level

print(f"\nHierarchy weights: code={ALPHA_CODE}, sub={ALPHA_SUB}, cat={ALPHA_CAT}")

# Store combined A for each group
A_combined = {}

for key, data in A_matrices.items():
    parts = key.split('_')
    code = parts[0]
    sub = parts[1]
    cat = parts[2]
    horizon = int(parts[3])

    sub_key = f"{code}_{sub}"

    # Get A from each level (fallback to global if not available)
    A_code = A_code_mean.get(code, np.zeros_like(data['A']))
    A_sub = A_sub_mean.get(sub_key, np.zeros_like(data['A']))
    A_cat = A_cat_mean.get(cat, np.zeros_like(data['A']))

    # Weighted combination
    A_combined[key] = ALPHA_CODE * A_code + ALPHA_SUB * A_sub + ALPHA_CAT * A_cat

print(f"✅ Combined hierarchical A for {len(A_combined)} groups")

# ============================================
# 7.3 Verify stability of combined A matrices
# ============================================

print("\nChecking stability of combined A matrices...")

def check_stability(A):
    """Check if all eigenvalues have negative real parts"""
    eigvals = np.linalg.eigvals(A)
    return np.all(np.real(eigvals) < 1e-6)  # stable if real parts < 0

stable_count = 0
unstable_count = 0

for key, A in A_combined.items():
    if check_stability(A):
        stable_count += 1
    else:
        unstable_count += 1
        if unstable_count <= 5:
            eigvals = np.linalg.eigvals(A)
            print(f"  ⚠️ Unstable A for {key}: max real part = {np.max(np.real(eigvals)):.4f}")

print(f"\nStability summary:")
print(f"  Stable matrices: {stable_count}")
print(f"  Unstable matrices: {unstable_count}")

if unstable_count > 0:
    print(f"\n⚠️ Warning: {unstable_count} groups have unstable dynamics")
    print("   This may cause exploding predictions. Consider:")
    print("   - Increasing regularization when estimating A")
    print("   - Using spectral normalization")
    print("   - Clipping eigenvalues > 0")

# ============================================
# 7.4 Store combined A for prediction
# ============================================

# Save A matrices for test prediction
A_for_prediction = {}

for key, data in A_matrices.items():
    parts = key.split('_')
    code = parts[0]
    sub = parts[1]
    cat = parts[2]
    horizon = int(parts[3])

    A_comb = A_combined[key]
    last_state = data['last_state']

    A_for_prediction[key] = {
        'A': A_comb,
        'last_state': last_state,
        'horizon': horizon,
        'code': code,
        'sub': sub,
        'cat': cat
    }

print(f"\n✅ Hierarchical A ready for prediction on {len(A_for_prediction)} groups")

## 8 BAYESIAN UNCERTAINTY

**Purpose:** Instead of single A matrix, sample from parameter distribution

**Implementation:**
- Use Bayesian Ridge to estimate parameter distributions
- Generate multiple A matrix samples
- Quantify prediction uncertainty

In [ ]:
# ============================================
# STEP 8: BAYESIAN UNCERTAINTY FOR A MATRIX
# ============================================
print("="*60)
print("STEP 8: BAYESIAN UNCERTAINTY QUANTIFICATION")
print("="*60)

from sklearn.linear_model import BayesianRidge
import scipy.stats as stats

# ============================================
# 8.1 Extract dominant eigenvalues as λ
# ============================================

print("\nExtracting dominant eigenvalues from A matrices...")

# For each group, compute dominant eigenvalue (λ) which captures main dynamics
lambda_by_group = {}

for key, data in A_for_prediction.items():
    A = data['A']

    # Compute eigenvalues
    eigvals = np.linalg.eigvals(A)

    # Dominant eigenvalue (largest real part)
    # This captures the main exponential growth/decay rate
    dominant_lambda = np.max(np.real(eigvals))

    # Also store all eigenvalues for uncertainty
    lambda_by_group[key] = {
        'dominant': dominant_lambda,
        'all_eigvals': eigvals,
        'spectral_radius': np.max(np.abs(eigvals)),
        'code': data['code'],
        'sub': data['sub'],
        'cat': data['cat'],
        'horizon': data['horizon']
    }

print(f"✅ Extracted dominant λ for {len(lambda_by_group)} groups")

# ============================================
# 8.2 Bayesian estimation of λ distribution per hierarchy level
# ============================================

print("\nEstimating Bayesian posterior for λ per level...")

# Collect λ by hierarchy level
lambda_by_code = {}
lambda_by_sub = {}
lambda_by_cat = {}

for key, data in lambda_by_group.items():
    code = data['code']
    sub = data['sub']
    cat = data['cat']
    lam = data['dominant']

    if code not in lambda_by_code:
        lambda_by_code[code] = []
    lambda_by_code[code].append(lam)

    sub_key = f"{code}_{sub}"
    if sub_key not in lambda_by_sub:
        lambda_by_sub[sub_key] = []
    lambda_by_sub[sub_key].append(lam)

    if cat not in lambda_by_cat:
        lambda_by_cat[cat] = []
    lambda_by_cat[cat].append(lam)

# Bayesian estimation: λ ~ Normal(μ, σ)
# Using empirical Bayes (conjugate prior)

def estimate_bayesian_lambda(values):
    """
    Estimate posterior distribution of λ given observed values
    Using Normal-Inverse-Gamma conjugate prior
    Returns: (mu, sigma, credible_interval)
    """
    if len(values) < 3:
        return np.mean(values), np.std(values), (np.mean(values)-0.1, np.mean(values)+0.1)

    # Empirical Bayes: prior from data
    mu_prior = np.mean(values)
    sigma_prior = np.std(values)

    # Posterior parameters
    n = len(values)
    mu_post = (mu_prior + n * np.mean(values)) / (1 + n)
    sigma_post = np.sqrt((sigma_prior**2 + n * np.var(values)) / (1 + n))

    # 95% credible interval
    ci_lower = mu_post - 1.96 * sigma_post / np.sqrt(n)
    ci_upper = mu_post + 1.96 * sigma_post / np.sqrt(n)

    return mu_post, sigma_post, (ci_lower, ci_upper)

# Compute Bayesian λ for each level
print("\nComputing Bayesian λ distributions...")

bayesian_lambda = {
    'code': {},
    'sub': {},
    'cat': {}
}

for code, lambdas in lambda_by_code.items():
    mu, sigma, ci = estimate_bayesian_lambda(lambdas)
    bayesian_lambda['code'][code] = {
        'mu': mu,
        'sigma': sigma,
        'ci': ci,
        'n': len(lambdas)
    }

for sub_key, lambdas in lambda_by_sub.items():
    mu, sigma, ci = estimate_bayesian_lambda(lambdas)
    bayesian_lambda['sub'][sub_key] = {
        'mu': mu,
        'sigma': sigma,
        'ci': ci,
        'n': len(lambdas)
    }

for cat, lambdas in lambda_by_cat.items():
    mu, sigma, ci = estimate_bayesian_lambda(lambdas)
    bayesian_lambda['cat'][cat] = {
        'mu': mu,
        'sigma': sigma,
        'ci': ci,
        'n': len(lambdas)
    }

print(f"✅ Bayesian λ distributions computed:")
print(f"   Code level: {len(bayesian_lambda['code'])} groups")
print(f"   Sub-code level: {len(bayesian_lambda['sub'])} groups")
print(f"   Category level: {len(bayesian_lambda['cat'])} groups")

# ============================================
# 8.3 Sample λ from posterior for prediction
# ============================================

print("\nPreparing posterior sampling for predictions...")

def sample_lambda_posterior(code, sub, cat, n_samples=100):
    """
    Sample λ from hierarchical posterior distribution
    """
    # Get distributions for each level
    code_dist = bayesian_lambda['code'].get(code, {'mu': 0.0, 'sigma': 0.1})
    sub_key = f"{code}_{sub}"
    sub_dist = bayesian_lambda['sub'].get(sub_key, {'mu': 0.0, 'sigma': 0.1})
    cat_dist = bayesian_lambda['cat'].get(cat, {'mu': 0.0, 'sigma': 0.1})

    # Sample from each level
    samples_code = np.random.normal(code_dist['mu'], code_dist['sigma'], n_samples)
    samples_sub = np.random.normal(sub_dist['mu'], sub_dist['sigma'], n_samples)
    samples_cat = np.random.normal(cat_dist['mu'], cat_dist['sigma'], n_samples)

    # Hierarchical weighting (same as A combination)
    weights = [ALPHA_CODE, ALPHA_SUB, ALPHA_CAT]
    samples_combined = weights[0] * samples_code + weights[1] * samples_sub + weights[2] * samples_cat

    return samples_combined

# Store sampling function for each group
lambda_samples_by_group = {}

for key, data in lambda_by_group.items():
    code = data['code']
    sub = data['sub']
    cat = data['cat']

    # Generate samples once
    samples = sample_lambda_posterior(code, sub, cat, n_samples=200)

    lambda_samples_by_group[key] = {
        'samples': samples,
        'mu': np.mean(samples),
        'sigma': np.std(samples),
        'code': code,
        'sub': sub,
        'cat': cat
    }

print(f"✅ Posterior samples prepared for {len(lambda_samples_by_group)} groups")

# ============================================
# 8.4 Uncertainty metrics
# ============================================

print("\nComputing uncertainty metrics...")

uncertainty_stats = []

for key, data in lambda_samples_by_group.items():
    samples = data['samples']

    # Coefficient of variation (relative uncertainty)
    cv = data['sigma'] / (abs(data['mu']) + 1e-8)

    # 95% credible interval width
    ci_width = np.percentile(samples, 97.5) - np.percentile(samples, 2.5)

    uncertainty_stats.append({
        'group': key,
        'mu': data['mu'],
        'sigma': data['sigma'],
        'cv': cv,
        'ci_width': ci_width
    })

# Sort by uncertainty
uncertainty_df = pl.DataFrame(uncertainty_stats).sort('cv', descending=True)

print("\nTop 10 groups with highest uncertainty:")
print(uncertainty_df.head(10))

print(f"\n✅ Bayesian uncertainty quantification complete")
print(f"   Average CV: {np.mean([u['cv'] for u in uncertainty_stats]):.4f}")
print(f"   Max CV: {np.max([u['cv'] for u in uncertainty_stats]):.4f}")

## 9 EXPONENTIAL FORECASTING

**Purpose:** Generate predictions using exponential state transitions

**Implementation:**
- For test data: X_pred = e^(A * h) * X_last
- Extract y_pred from state vector
- Use hierarchical A matrices with uncertainty

In [ ]:
# ============================================
# STEP 9: EXPONENTIAL FORECAST WITH SAMPLING
# ============================================
print("="*60)
print("STEP 9: EXPONENTIAL FORECAST FROM STATE SPACE")
print("="*60)

# ============================================
# 9.1 Prepare test data groups
# ============================================

print("\nPreparing test data groups...")

# Group test data by code, sub_code, sub_category, horizon
test_groups = test_full.group_by(['code', 'sub_code', 'sub_category', 'horizon'])

# Store predictions per group
exp_predictions = {}

for (code, sub, cat, horizon), group in test_groups:
    group = group.sort('ts_index')

    # Find matching trained group
    group_key = f"{code}_{sub}_{cat}_{horizon}"

    if group_key not in lambda_samples_by_group:
        print(f"  ⚠️ No trained model for {group_key}, using fallback")
        exp_predictions[group_key] = None
        continue

    # Get last observed y_target from train for this group
    # Need to find last train value for this exact group
    train_group = train_full.filter(
        (pl.col('code') == code) &
        (pl.col('sub_code') == sub) &
        (pl.col('sub_category') == cat) &
        (pl.col('horizon') == horizon)
    ).sort('ts_index')

    if len(train_group) == 0:
        print(f"  ⚠️ No train data for {group_key}, using fallback")
        exp_predictions[group_key] = None
        continue

    y_last = train_group.select('y_target').to_numpy()[-1][0]
    lambda_samples = lambda_samples_by_group[group_key]['samples']

    exp_predictions[group_key] = {
        'y_last': y_last,
        'lambda_samples': lambda_samples,
        'horizon': horizon,
        'code': code,
        'sub': sub,
        'cat': cat,
        'n_samples': len(lambda_samples)
    }

print(f"✅ Prepared predictions for {len([k for k,v in exp_predictions.items() if v is not None])} groups")

# ============================================
# 9.2 Generate forecasts with uncertainty
# ============================================

print("\nGenerating exponential forecasts...")

def exponential_forecast_with_uncertainty(y_last, lambda_samples, horizon):
    """
    Generate forecasts using sampled λ
    Returns: mean_pred, std_pred, all_samples
    """
    # y(t+h) = y(t) * exp(λ * h)
    forecasts = y_last * np.exp(lambda_samples * horizon)

    return np.mean(forecasts), np.std(forecasts), forecasts

# Store forecasts
forecasts_by_group = {}

for group_key, data in exp_predictions.items():
    if data is None:
        continue

    y_last = data['y_last']
    lambda_samples = data['lambda_samples']
    horizon = data['horizon']

    mean_pred, std_pred, all_samples = exponential_forecast_with_uncertainty(
        y_last, lambda_samples, horizon
    )

    forecasts_by_group[group_key] = {
        'mean': mean_pred,
        'std': std_pred,
        'samples': all_samples,
        'y_last': y_last,
        'lambda_mu': np.mean(lambda_samples),
        'lambda_sigma': np.std(lambda_samples),
        'horizon': horizon,
        'code': data['code'],
        'sub': data['sub'],
        'cat': data['cat']
    }

    if len(forecasts_by_group) % 500 == 0:
        print(f"  Processed {len(forecasts_by_group)} groups")

print(f"✅ Generated forecasts for {len(forecasts_by_group)} groups")

# ============================================
# 9.3 Apply to test rows
# ============================================

print("\nApplying forecasts to test rows...")

# Create prediction column in test dataframe
test_full = test_full.with_columns([
    pl.lit(0.0).alias('exp_prediction'),
    pl.lit(0.0).alias('exp_uncertainty')
])

# For each test row, assign prediction from its group
for group_key, forecast in forecasts_by_group.items():
    code, sub, cat, horizon = group_key.split('_')
    horizon = int(horizon)

    # Find rows in test with this group
    mask = (
        (test_full['code'] == code) &
        (test_full['sub_code'] == sub) &
        (test_full['sub_category'] == cat) &
        (test_full['horizon'] == horizon)
    )

    if mask.sum() == 0:
        continue

    # Update predictions
    test_full = test_full.with_columns([
        pl.when(mask)
        .then(pl.lit(forecast['mean']))
        .otherwise(pl.col('exp_prediction'))
        .alias('exp_prediction'),

        pl.when(mask)
        .then(pl.lit(forecast['std']))
        .otherwise(pl.col('exp_uncertainty'))
        .alias('exp_uncertainty')
    ])

print(f"✅ Predictions assigned to test rows")

# ============================================
# 9.4 Fallback for groups without model
# ============================================

# Check for any rows without predictions
null_preds = test_full.filter(pl.col('exp_prediction') == 0.0)
if len(null_preds) > 0:
    print(f"\n⚠️ {len(null_preds)} rows without exponential prediction")
    print("Using forward fill fallback...")

    # Use last known y_target from same sub_category as fallback
    fallback_preds = []
    for row in null_preds.iter_rows(named=True):
        # Find similar group in train
        similar_train = train_full.filter(
            (pl.col('sub_category') == row['sub_category']) &
            (pl.col('horizon') == row['horizon'])
        ).sort('ts_index', descending=True).head(1)

        if len(similar_train) > 0:
            fallback = similar_train.select('y_target').to_numpy()[0][0]
        else:
            fallback = 0.0

        fallback_preds.append(fallback)

    # Update fallback rows
    null_ids = null_preds.select('id').to_numpy().ravel()
    for idx, row_id in enumerate(null_ids):
        test_full = test_full.with_columns([
            pl.when(pl.col('id') == row_id)
            .then(pl.lit(fallback_preds[idx]))
            .otherwise(pl.col('exp_prediction'))
            .alias('exp_prediction')
        ])

print(f"✅ All test rows have predictions")

# ============================================
# 9.5 Summary of exponential predictions
# ============================================

print("\n" + "="*60)
print("EXPONENTIAL FORECAST SUMMARY")
print("="*60)

# Statistics by horizon
for horizon in [1, 3, 10, 25]:
    horizon_preds = test_full.filter(pl.col('horizon') == horizon).select('exp_prediction').to_numpy().ravel()
    horizon_uncertainty = test_full.filter(pl.col('horizon') == horizon).select('exp_uncertainty').to_numpy().ravel()

    print(f"\nHorizon {horizon}:")
    print(f"  Mean prediction: {np.mean(horizon_preds):.6f}")
    print(f"  Std prediction: {np.std(horizon_preds):.6f}")
    print(f"  Mean uncertainty: {np.mean(horizon_uncertainty):.6f}")
    print(f"  Range: [{np.min(horizon_preds):.4f}, {np.max(horizon_preds):.4f}]")

# Show sample predictions
print("\nSample predictions:")
sample_rows = test_full.select(['id', 'horizon', 'exp_prediction', 'exp_uncertainty']).head(10)
print(sample_rows)

print(f"\n✅ Exponential forecasts ready for submission")

## 10 GARCH ON RESIDUALS

**Purpose:** Model residual volatility and calibrate predictions

**Implementation:**
- Residual = y_true - y_pred (on train)
- Train GARCH(1,1)
- Calibrate predictions

In [ ]:
# ============================================
# STEP 10: GARCH VOLATILITY MODELLING
# ============================================
print("="*60)
print("STEP 10: GARCH(1,1) FOR RESIDUAL VOLATILITY")
print("="*60)

try:
    from arch import arch_model
    HAS_ARCH = True
    print("✅ arch package available")
except ImportError:
    HAS_ARCH = False
    print("⚠️ arch package not available. Install with: pip install arch")
    print("Skipping GARCH step...")

# ============================================
# 10.1 Compute residuals on train
# ============================================

if HAS_ARCH:
    print("\nComputing residuals on training data...")

    # Create predictions for train using exponential model
    train_with_exp = train_full.clone()
    train_with_exp = train_with_exp.with_columns([
        pl.lit(0.0).alias('exp_prediction_train')
    ])

    # For each group in train, compute exponential prediction
    train_groups = train_full.group_by(['code', 'sub_code', 'sub_category', 'horizon'])

    for (code, sub, cat, horizon), group in train_groups:
        group_key = f"{code}_{sub}_{cat}_{horizon}"

        if group_key not in forecasts_by_group:
            continue

        # Get λ samples for this group
        lambda_samples = lambda_samples_by_group[group_key]['samples']

        # For each row in group, compute prediction based on previous y
        group_sorted = group.sort('ts_index')
        y_values = group_sorted.select('y_target').to_numpy().ravel()

        predictions = []
        for i, y_current in enumerate(y_values):
            if i == 0:
                # First row: use group mean or forward fill
                pred = np.mean(y_values[:5]) if len(y_values) >= 5 else y_current
            else:
                # Predict next from previous: y_prev * exp(λ * horizon)
                # But here horizon is fixed per group
                y_prev = y_values[i-1]
                # Use mean λ for deterministic prediction
                lambda_mu = np.mean(lambda_samples)
                pred = y_prev * np.exp(lambda_mu * horizon)
            predictions.append(pred)

        predictions = np.array(predictions)

        # Update train_with_exp
        ids = group_sorted.select('id').to_numpy().ravel()
        for idx, row_id in enumerate(ids):
            train_with_exp = train_with_exp.with_columns([
                pl.when(pl.col('id') == row_id)
                .then(pl.lit(predictions[idx]))
                .otherwise(pl.col('exp_prediction_train'))
                .alias('exp_prediction_train')
            ])

    # Compute residuals
    train_with_exp = train_with_exp.with_columns([
        (pl.col('y_target') - pl.col('exp_prediction_train')).alias('residual')
    ])

    print(f"✅ Residuals computed for {len(train_with_exp):,} rows")

    # ============================================
    # 10.2 Train GARCH per horizon
    # ============================================

    print("\nTraining GARCH(1,1) models per horizon...")

    garch_models = {}
    garch_forecasts = {}

    for horizon in [1, 3, 10, 25]:
        print(f"\n  Horizon {horizon}:")

        # Get residuals for this horizon
        residuals_h = train_with_exp.filter(pl.col('horizon') == horizon).select('residual').to_numpy().ravel()

        # Remove zeros and extreme values for stability
        residuals_h = residuals_h[~np.isnan(residuals_h)]
        residuals_h = residuals_h[np.abs(residuals_h) < np.percentile(np.abs(residuals_h), 99)]

        if len(residuals_h) < 100:
            print(f"    ⚠️ Not enough data ({len(residuals_h)}), skipping")
            garch_models[horizon] = None
            continue

        try:
            # Fit GARCH(1,1)
            model = arch_model(residuals_h * 100, vol='Garch', p=1, q=1, dist='normal')
            garch_fit = model.fit(disp='off')

            garch_models[horizon] = garch_fit

            # Forecast volatility for test
            forecast = garch_fit.forecast(horizon=1)
            volatility_forecast = np.sqrt(forecast.variance.values[-1, 0]) / 100  # rescale back

            garch_forecasts[horizon] = {
                'volatility': volatility_forecast,
                'params': garch_fit.params
            }

            print(f"    ✅ GARCH trained")
            print(f"       Omega: {garch_fit.params['omega']:.6f}")
            print(f"       Alpha: {garch_fit.params['alpha[1]']:.6f}")
            print(f"       Beta: {garch_fit.params['beta[1]']:.6f}")
            print(f"       Forecast volatility: {volatility_forecast:.6f}")

        except Exception as e:
            print(f"    ❌ GARCH failed: {e}")
            garch_models[horizon] = None

    # ============================================
    # 10.3 Apply GARCH calibration to predictions
    # ============================================

    print("\nApplying GARCH calibration to test predictions...")

    # Copy test predictions
    test_full = test_full.with_columns([
        pl.col('exp_prediction').alias('exp_prediction_calibrated')
    ])

    for horizon in [1, 3, 10, 25]:
        if garch_models[horizon] is None:
            continue

        # Get volatility forecast
        vol_forecast = garch_forecasts[horizon]['volatility']

        # Adjust predictions: add volatility component
        # Idea: y_calibrated = y_exp * (1 + α * volatility)
        # Higher volatility = wider prediction range
        alpha = 0.1  # tunable parameter

        mask = test_full['horizon'] == horizon

        test_full = test_full.with_columns([
            pl.when(mask)
            .then(pl.col('exp_prediction') * (1 + alpha * vol_forecast))
            .otherwise(pl.col('exp_prediction_calibrated'))
            .alias('exp_prediction_calibrated')
        ])

        print(f"  Horizon {horizon}: volatility adjustment = {alpha * vol_forecast:.6f}")

    print(f"✅ GARCH calibration applied")

else:
    print("\n⚠️ GARCH step skipped (arch package not available)")
    # Copy original predictions as fallback
    test_full = test_full.with_columns([
        pl.col('exp_prediction').alias('exp_prediction_calibrated')
    ])

# ============================================
# 10.4 Summary
# ============================================

print("\n" + "="*60)
print("GARCH CALIBRATION SUMMARY")
print("="*60)

for horizon in [1, 3, 10, 25]:
    if HAS_ARCH and garch_models.get(horizon) is not None:
        original = test_full.filter(pl.col('horizon') == horizon).select('exp_prediction').to_numpy().ravel()
        calibrated = test_full.filter(pl.col('horizon') == horizon).select('exp_prediction_calibrated').to_numpy().ravel()

        print(f"\nHorizon {horizon}:")
        print(f"  Original mean: {np.mean(original):.6f}")
        print(f"  Calibrated mean: {np.mean(calibrated):.6f}")
        print(f"  Change: {np.mean(calibrated) - np.mean(original):.6f}")
    else:
        print(f"\nHorizon {horizon}: GARCH not applied")

print(f"\n✅ Step 10 complete")

## 10.1 ENSEMBLE WITH LIGHTGBM V4

**Purpose:** Combine exponential model with LightGBM v4 predictions

**Implementation:**
- Load LightGBM v4 predictions
- Optimal ensemble weights: y_final = w1*y_exp + w2*y_lgb
- Weight optimization based on validation performance

## 11.2 SAVE SUBMISSION

In [ ]:
# ============================================
# STEP 11: SUBMISSION FROM EXPONENTIAL MODEL
# ============================================
print("="*60)
print("STEP 11: GENERATING SUBMISSION")
print("="*60)

# ============================================
# 11.1 Prepare submission dataframe
# ============================================

# Use calibrated predictions (or fallback to original if GARCH skipped)
if 'exp_prediction_calibrated' in test_full.columns:
    submission_df = test_full.select(['id', 'exp_prediction_calibrated']).rename({'exp_prediction_calibrated': 'prediction'})
else:
    submission_df = test_full.select(['id', 'exp_prediction']).rename({'exp_prediction': 'prediction'})

print(f"Submission shape: {submission_df.shape}")
print("\nSample predictions:")
print(submission_df.head(10))

# ============================================
# 11.2 Basic statistics
# ============================================

print("\n" + "="*60)
print("PREDICTION STATISTICS")
print("="*60)

preds = submission_df.select('prediction').to_numpy().ravel()

print(f"Total predictions: {len(preds):,}")
print(f"Mean: {np.mean(preds):.6f}")
print(f"Std: {np.std(preds):.6f}")
print(f"Min: {np.min(preds):.6f}")
print(f"Max: {np.max(preds):.6f}")
print(f"Median: {np.median(preds):.6f}")

# By horizon
for horizon in [1, 3, 10, 25]:
    horizon_ids = test_full.filter(pl.col('horizon') == horizon).select('id').to_numpy().ravel()
    horizon_preds = submission_df.filter(pl.col('id').is_in(horizon_ids)).select('prediction').to_numpy().ravel()

    print(f"\nHorizon {horizon}:")
    print(f"  Count: {len(horizon_preds):,}")
    print(f"  Mean: {np.mean(horizon_preds):.6f}")
    print(f"  Std: {np.std(horizon_preds):.6f}")
    print(f"  Range: [{np.min(horizon_preds):.4f}, {np.max(horizon_preds):.4f}]")

# ============================================
# 11.3 Save submission
# ============================================

import datetime
from pathlib import Path

# Create results directory if needed
results_dir = base_dir / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

# Generate filename
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"exponential_model_v5_{timestamp}.csv"
submission_path = results_dir / submission_filename

# Save
submission_df.write_csv(submission_path)

print(f"\n" + "="*60)
print("SUBMISSION SAVED")
print("="*60)
print(f"File: {submission_path}")
print(f"Size: {submission_path.stat().st_size / 1024**2:.2f} MB")
print(f"Rows: {len(submission_df):,}")

# ============================================
# 11.4 Optional: Validation on train split
# ============================================

print("\n" + "="*60)
print("VALIDATION ON TRAIN (ts_index > 3500)")
print("="*60)

# Split train into fit (ts_index <= 3500) and val (ts_index > 3500)
train_fit = train_full.filter(pl.col('ts_index') <= 3500)
train_val = train_full.filter(pl.col('ts_index') > 3500)

if len(train_val) > 0:
    print(f"Validation rows: {len(train_val):,}")

    # We need to compute exponential predictions for validation
    # This would require running the same exponential model on train_val
    # For now, just note that this is possible
    print("Note: To compute validation score, run exponential model on train_val")
    print("      using the same λ distributions from train_fit.")

    # Simplified validation using existing forecasts_by_group
    val_predictions = []
    val_actuals = []
    val_weights = []

    for row in train_val.iter_rows(named=True):
        group_key = f"{row['code']}_{row['sub_code']}_{row['sub_category']}_{row['horizon']}"

        if group_key in forecasts_by_group:
            # Use the mean prediction from the group
            pred = forecasts_by_group[group_key]['mean']
            val_predictions.append(pred)
            val_actuals.append(row['y_target'])
            val_weights.append(row['weight'])

    if len(val_predictions) > 0:
        val_predictions = np.array(val_predictions)
        val_actuals = np.array(val_actuals)
        val_weights = np.array(val_weights)

        val_score = weighted_rmse_score(val_actuals, val_predictions, val_weights)
        print(f"\nValidation Score (exponential model): {val_score:.6f}")
        print(f"Note: This is on {len(val_predictions):,} rows from train_val")
    else:
        print("Not enough validation predictions to compute score")
else:
    print("No validation rows (ts_index > 3500)")

print("\n" + "="*60)
print("✅ EXPONENTIAL MODEL v5 COMPLETE")
print("="*60)
print(f"\n🎯 Submission ready: {submission_filename}")
print(f"📊 Model type: Hierarchical Exponential with Bayesian λ + GARCH")
print(f"📈 Ready to upload to Kaggle!")